# ТЕМА 5. Екосистема LangChain

У цьому уроці ми розглянемо основні можливості LangChain: створення ланцюгів (chains), роботу з промптами та структурованим виходом. LangChain дозволяє будувати складні AI-застосунки через композицію простих компонентів.

> **Версія.** Код розрахований на **LangChain 0.3** (лінія курсу). Імпорти йдуть з `langchain_core`, тому той самий запис працює і на 1.x.
>
> **Довідник до цього ноутбука:** `5_langchain_ecosystem_notes.md` у цій самій папці. Паралельні гілки, `RunnableBranch`, трасування LangSmith і наскрізний кейс є в розширеному ноутбуці `MU of AI_topic_5_lesson.ipynb`.


### Встановлення пакетів

Фіксуємо узгоджений набір лінії 0.3. Діапазон `langchain==0.3.*` цього не гарантує: розв'язувач може поставити `langchain-openai` 1.x поруч із ядром 0.3.

`langgraph` і `faiss-cpu` в цій темі не використовуються. Їх ставлять наступні теми, окремо, щоб вони не потягнули іншу лінію LangChain.

**У Google Colab:** якщо після встановлення з'явиться попередження про перезапуск, зробіть *Runtime → Restart session* і запустіть ноутбук спочатку.


In [ ]:
# Узгоджені версії, на яких перевірено конспект.
# langchain-core 0.3.15 із роздатки не ставимо: він несумісний із langchain-openai 0.3.x.
!pip install -q "langchain==0.3.30" "langchain-core==0.3.86" "langchain-openai==0.3.35" "langchain-community==0.3.31" "openai==1.109.1" python-dotenv


### Налаштування API ключів

Ключ не пишемо літералом у клітинку: ноутбук часто комітять. `ChatOpenAI` сам читає лише `OPENAI_API_KEY` і `OPENAI_BASE_URL`, тому ключ OpenRouter треба отримати явно і передати в конструктор.

Порядок у `get_secret`: змінна середовища (або `.env`, якщо ви вже викликали `load_dotenv()`), потім секрети Colab, потім приховане поле вводу.


In [ ]:
import os
from getpass import getpass


def get_secret(name: str) -> str:
    """Повертає секрет, не друкуючи його у вивід ноутбука.

    Спочатку дивиться змінну середовища, потім секрети Google Colab.
    Якщо обох немає, питає значення через getpass.
    """
    if os.environ.get(name):
        return os.environ[name]
    try:
        from google.colab import userdata

        value = userdata.get(name)
        if value:
            return value
    except Exception:
        # Локальний Jupyter не має google.colab. Це очікувано, не помилка ключа.
        pass
    return getpass(f"Введіть {name}: ")


API_KEY = get_secret("OPENROUTER_API_KEY")
# Адреса без кутових дужок. Дужки в роздатці були артефактом автопосилання.
BASE_URL = "https://openrouter.ai/api/v1"
# Префікс openai/ обов'язковий на OpenRouter. Голе gpt-4o-mini пішло б на API OpenAI.
MODEL = "openai/gpt-4o-mini"


### 1. Базовий приклад з LangChain

Ініціалізуємо модель `ChatOpenAI` через OpenRouter. Це ще не ланцюг: один клієнт, один виклик.


In [ ]:
from langchain_openai import ChatOpenAI

# api_key і base_url передаємо явно. Змінну OPENROUTER_API_KEY конструктор не підхоплює.
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model=MODEL,
)


Виконуємо простий запит. `.invoke()` повертає `AIMessage`, текст відповіді лежить у `.content`.


In [ ]:
# Рядок на вході чат-модель приймає як одне людське повідомлення.
resp = llm.invoke("What is LangChain?")
print(resp.content)


LangChain is an open-source framework designed to facilitate the development of applications using large language models (LLMs). It provides tools and components to streamline the building of applications that leverage these models for various tasks, including text generation, question-answering, summarization, chatbots, and more.

Key features of LangChain include:

1. **Modularity**: LangChain is built with a modular architecture, allowing developers to easily mix and match components such as model providers, memory management, and prompt templates.

2. **Integration with Various Data Sources**: The framework can connect to various data sources, enabling applications to pull in relevant information from APIs, databases, or other data stores.

3. **Prompt Management**: LangChain helps with crafting, managing, and optimizing prompts, which are crucial for eliciting desired responses from language models.

4. **Chain Management**: It supports the creation of chains, where the output of 

### 2. Шаблони промптів (Prompt Templates)

Створюємо шаблон промпта зі змінною.

Імпорт беремо з `langchain_core.prompts`. Шлях `langchain.prompts` на 0.3 ще реекспортується, на 1.x його прибрали.


In [ ]:
from langchain_core.prompts import PromptTemplate

# input_variables має збігатися з іменами в {дужках}. Інакше format впаде на відсутній ключ.
template = PromptTemplate(
    input_variables=["topic"],
    template="Explain {topic} in simple terms.",
)


Підставляємо значення. `format` повертає **рядок**. Чат-ланцюг у наступному розділі так не робить: там шаблон одразу віддає список повідомлень.


In [ ]:
# format підставляє topic і повертає один рядок, ще не список повідомлень.
prompt = template.format(topic="Artificial Intelligence")
print(prompt)


Explain Artificial Intelligence in simple terms.


### 3. Створення ланцюгів (Chains) з LCEL

Налаштовуємо модель з `temperature=0.2`, щоб повторні запуски того самого ланцюга менше розходились.


In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model=MODEL,
    temperature=0.2,  # 0 — майже детерміновано, 1 — помітний розкид
)


Створюємо ланцюг: промпт → модель → парсер.

Оператор `|` викликає `Runnable.__or__`. Вихід лівого кроку стає входом правого без ручного клею.


In [ ]:
# from_template збирає одне людське повідомлення, а не голий рядок.
prompt = ChatPromptTemplate.from_template("Explain {topic} in simple terms.")

# Промпт приймає dict, модель повертає AIMessage, парсер залишає str.
chain = prompt | llm | StrOutputParser()


Запускаємо ланцюг. Ключ словника має називатися `topic`, як у шаблоні. Друкарська помилка на кшталт `topik` падає ще до моделі.


In [ ]:
# На вхід — словник змінних шаблону, на вихід — рядок після парсера.
print(chain.invoke({"topic": "Neural Networks"}))


Sure! Think of a neural network as a system that mimics how our brains work to help computers learn and make decisions.

1. **Basic Structure**: A neural network is made up of layers of nodes, which are like tiny processing units. The first layer takes in input data (like images, text, or numbers), and the last layer produces the output (like a classification or prediction).

2. **Layers**: 
   - **Input Layer**: This is where the data enters the network.
   - **Hidden Layers**: These are the layers in between the input and output. They do the heavy lifting by processing the data through various transformations.
   - **Output Layer**: This is where the final result comes out, telling us what the network has learned.

3. **Connections**: Each node in one layer is connected to nodes in the next layer. These connections have weights, which determine how much influence one node has on another. During training, the network adjusts these weights to improve its accuracy.

4. **Learning Proces

### 4. Структурований вихід з `.with_structured_output()`

#### 4.1. Використання Pydantic (з валідацією)

Ініціалізуємо модель ще раз, уже без зниженої температури: далі відповідь тримає схема, а не температура.


In [ ]:
from typing import Optional

from pydantic import BaseModel, Field

# Імпорти знадобляться наступній клітинці: клас Joke живе в спільному просторі ноутбука.
llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model=MODEL,
)


Описуємо структуру жарту через Pydantic.

Docstring класу і `description` полів потрапляють у схему, яку бачить модель. Це інструкція для заповнення, не коментар «для людей». Межі `ge`/`le` тут навмисно не ставимо: їх показує розділ про retry, щоб два приклади не розходились із уже збереженим виводом.


In [ ]:
class Joke(BaseModel):
    """Жарт із підводкою, панчлайном і необов'язковою оцінкою.

    Модель має повернути саме ці поля, а не абзац вільного тексту.
    """

    setup: str = Field(description="Підводка до жарту")
    punchline: str = Field(description="Панчлайн")
    rating: Optional[int] = Field(
        default=None,
        description="Оцінка смішності від 1 до 10",
    )


Отримуємо екземпляр `Joke`, не рядок. У `langchain-openai` 0.3.35 метод за замовчуванням — `json_schema`: провайдер тримає відповідь у межах схеми.


In [ ]:
structured_llm = llm.with_structured_output(Joke)
result = structured_llm.invoke("Tell me a joke about cats")

# Атрибути читаються як у звичайного об'єкта. model_dump() — словник Pydantic 2.
print(result.setup)
print(result.punchline)
print(result.rating)
print("\nЯк словник:", result.model_dump())


Why did the cat sit on the computer?
Because it wanted to keep an eye on the mouse!
7

Як словник: {'setup': 'Why did the cat sit on the computer?', 'punchline': 'Because it wanted to keep an eye on the mouse!', 'rating': 7}


#### 4.2. Використання TypedDict (для стримінгу)

Описуємо ту саму структуру через TypedDict.

Це шлях для словника і для стримінгу. Перевірки типів немає: другий аргумент `Annotated` лише вирішує, чи поле обов'язкове (`...` — так, `None` — ні), а рядок стає описом у схемі.


In [ ]:
from typing import Optional

from typing_extensions import Annotated, TypedDict


class JokeDict(TypedDict):
    """Жарт без валідації Pydantic.

    Якщо модель пропустить rating, ключ може просто відсутній.
    Ніхто не підставить None і не перевірить діапазон 1..10.
    """

    setup: Annotated[str, ..., "Підводка до жарту"]
    punchline: Annotated[str, ..., "Панчлайн"]
    rating: Annotated[Optional[int], None, "Оцінка смішності 1..10"]


На виході звичайний `dict`.


In [ ]:
# TypedDict-схема повертає dict. Перевірки меж rating тут немає.
structured_llm = llm.with_structured_output(JokeDict)
res = structured_llm.invoke("Tell me a joke about cats")
print(res)


{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it wanted to keep an eye on the mouse!', 'rating': 8}


#### 4.3. Використання JSON Schema

Та сама форма чистим JSON Schema, без класу Python. Так схему збирають динамічно. Список `required` вирішує обов'язковість; ключ `default` у властивості модель ні до чого не зобов'язує, тому його тут немає.


In [ ]:
json_schema = {
    "title": "joke",  # ім'я інструмента в схемі, яку бачить модель
    "description": "Структура жарту.",
    "type": "object",
    "properties": {
        "setup": {"type": "string", "description": "Підводка до жарту"},
        "punchline": {"type": "string", "description": "Панчлайн"},
        "rating": {
            "type": "integer",
            "description": "Оцінка 1..10",
        },
    },
    "required": ["setup", "punchline"],
}


Виклик повертає словник, як і TypedDict.


In [ ]:
# JSON Schema теж повертає dict: класу Python на виході немає.
structured_llm = llm.with_structured_output(json_schema)
res = structured_llm.invoke("Tell me a joke about cats")
print(res)


{'setup': 'Why did the cat sit on the computer?', 'punchline': 'Because it wanted to keep an eye on the mouse!', 'rating': 8}


#### 4.4. Union-схеми (кілька варіантів відповіді)

Два варіанти відповіді через Pydantic. Батьківська схема має одне поле `final_output`, і модель обирає рівно один із типів. Це вже перевіряється: чужа форма не пройде `ValidationError`.


In [ ]:
from typing import Optional, Union

from pydantic import BaseModel, Field


class Joke(BaseModel):
    """Жарт як один із двох дозволених варіантів відповіді."""

    setup: str = Field(description="Підводка")
    punchline: str = Field(description="Панчлайн")
    rating: Optional[int] = Field(default=None, description="Оцінка 1..10")


class ConversationalResponse(BaseModel):
    """Звичайна репліка, коли жарт недоречний."""

    response: str = Field(description="Текст розмовної відповіді")


class FinalResponse(BaseModel):
    """Обгортка: рівно один варіант у полі final_output."""

    final_output: Union[Joke, ConversationalResponse]


Той самий Runnable, два різні запити. Перший має зібратись у `Joke`, другий — у `ConversationalResponse`.


In [ ]:
# Один Runnable, два запити. Pydantic вибирає Joke або ConversationalResponse.
structured_llm = llm.with_structured_output(FinalResponse)

print("Запит жарту:")
print(structured_llm.invoke("Tell me a joke about cats"))

print("\nЗвичайне питання:")
print(structured_llm.invoke("How are you today?"))


Запит жарту:
final_output=Joke(setup='Why was the cat sitting on the computer?', punchline='Because it wanted to keep an eye on the mouse!', rating=7)

Звичайне питання:
final_output=ConversationalResponse(response="I'm just a computer program, but I'm here and ready to help you! How about you?")


Той самий Union через TypedDict. Перевірки немає: якщо модель пропустить обгортку `final_output`, винятку не буде. Збережений вивід клітинки нижче це вже показує — друга відповідь прийшла як `{"response": ...}` без `final_output`.


In [ ]:
from typing import Optional, Union

from typing_extensions import Annotated, TypedDict


class Joke(TypedDict):
    """Жарт у Union без валідації. Поля ті самі, що в Pydantic-варіанті."""

    setup: Annotated[str, ..., "Підводка"]
    punchline: Annotated[str, ..., "Панчлайн"]
    rating: Annotated[Optional[int], None, "Оцінка 1..10"]


class ConversationalResponse(TypedDict):
    """Репліка без структури жарту."""

    response: Annotated[str, ..., "Дружня відповідь"]


class FinalResponse(TypedDict):
    """Очікувана обгортка. TypedDict не змусить модель її дотриматись."""

    final_output: Union[Joke, ConversationalResponse]


structured_llm = llm.with_structured_output(FinalResponse)

print(structured_llm.invoke("Tell me a joke about cats"))
print(structured_llm.invoke("How are you today?"))


{'final_output': {'setup': 'What do you call a pile of cats?', 'punchline': 'A meowtain!', 'rating': 7}}
{'response': "I'm just a program, so I don't have feelings, but I'm here and ready to help you! How can I assist you today?"}


> **Що вже видно у виводі вище.** Другий виклик повернув `{"response": ...}` без ключа `final_output`, і код не впав. TypedDict таку відповідь приймає. Якщо від форми залежить гілка програми, цю схему треба лишати на Pydantic, як у клітинці перед цією.


#### 4.5. Стримінг структурованих відповідей

Стримінг структурованої відповіді працює для словника (TypedDict або JSON Schema), не для екземпляра Pydantic.

Кожен `chunk` — це вже зібраний частковий словник, а не дельта символів. Наступний шматок замінює попередній цілком. Pydantic-об'єкт так не тече: валідація чекає повну структуру.


In [ ]:
from typing import Optional

from typing_extensions import Annotated, TypedDict


class Joke(TypedDict):
    """Схема для стримінгу. На виході dict, тому потік взагалі можливий."""

    setup: Annotated[str, ..., "Підводка"]
    punchline: Annotated[str, ..., "Панчлайн"]
    rating: Annotated[Optional[int], None, "Оцінка 1..10"]


structured_llm = llm.with_structured_output(Joke)

# Перші шматки містять лише setup, далі з'являється punchline, наприкінці rating.
for chunk in structured_llm.stream("Tell me a joke about cats"):
    print(chunk)


{}
{'setup': ''}
{'setup': 'Why'}
{'setup': 'Why was'}
{'setup': 'Why was the'}
{'setup': 'Why was the cat'}
{'setup': 'Why was the cat sitting'}
{'setup': 'Why was the cat sitting on'}
{'setup': 'Why was the cat sitting on the'}
{'setup': 'Why was the cat sitting on the computer'}
{'setup': 'Why was the cat sitting on the computer?'}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': ''}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because'}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it'}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it wanted'}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it wanted to'}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it wanted to keep'}
{'setup': 'Why was the cat sitting on the computer?', 'punchline': 'Because it wanted to keep an'}
{'setup': 'Why was the cat sitting on the computer?', 'punc

#### 4.6. Few-shot: приклади в промпті

У системне повідомлення кладемо вже розкладені приклади. Фігурні дужки JSON подвоєні (`{{` і `}}`): інакше `ChatPromptTemplate` сприйме `{setup}` як свою змінну і впаде ще до моделі.


In [ ]:
from typing import Optional

from typing_extensions import Annotated, TypedDict
from langchain_core.prompts import ChatPromptTemplate

# Подвоєні дужки — літеральний JSON у тексті промпта, не змінні шаблону.
system = """Ти — дотепний комік. Поверни жарт з полями: setup, punchline, rating.

example_user: Tell me a joke about planes
example_assistant: {{"setup": "Why don't planes ever get tired?", "punchline": "Because they have rest wings!", "rating": 2}}

example_user: Tell me another joke about planes
example_assistant: {{"setup": "Cargo", "punchline": "Cargo 'vroom vroom', but planes go 'zoom zoom'!", "rating": 10}}

example_user: Now about caterpillars
example_assistant: {{"setup": "Caterpillar", "punchline": "Caterpillar really slow, but watch me turn into a butterfly and steal the show!", "rating": 5}}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system),
    ("human", "{input}"),  # єдина справжня змінна цього шаблону
])


Промпт і структуровану модель з'єднує той самий `|`. На вхід іде словник із ключем `input`.


In [ ]:
class Joke(TypedDict):
    """Цільова форма few-shot відповіді. Опис полів іде в схему разом із прикладами з промпта."""

    setup: Annotated[str, ..., "Підводка"]
    punchline: Annotated[str, ..., "Панчлайн"]
    rating: Annotated[Optional[int], None, "Оцінка 1..10"]


llm = ChatOpenAI(
    api_key=API_KEY,
    base_url=BASE_URL,
    model=MODEL,
)

few_shot_structured_llm = prompt | llm.with_structured_output(Joke)
print(few_shot_structured_llm.invoke({"input": "what's something funny about woodpeckers"}))


{'setup': 'Why did the woodpecker get kicked out of the party?', 'punchline': "Because he just couldn't stop pecking around for attention!", 'rating': 7}


#### 4.7. Обробка помилок валідації

`ValidationError` виникає лише для Pydantic. Клас `Joke` вище перезаписаний як TypedDict, тому ловити помилку на ньому немає сенсу: перевірка не запускається, і збережений вивід старої версії цієї клітинки був словником.

Нижче окремий клас `JokeModel` з межами оцінки. Додатково ловимо `OutputParserException`: його кидає парсер, коли відповідь узагалі не розібралась як об'єкт.


Повтор має сенс, коли схема справді відхилила відповідь. На `method="json_schema"` (типовий для цієї версії) модель рідко відхиляється від типів, тож цикл може жодного разу не зайти в `except`. Це не поламаний retry, а наслідок strict-схеми. Надійно побачити `ValidationError` можна прямим викликом `JokeModel(setup="...", punchline="...", rating=99)`.


In [ ]:
from typing import Optional

from langchain_core.exceptions import OutputParserException
from pydantic import BaseModel, Field, ValidationError


class JokeModel(BaseModel):
    """Жарт із перевіркою оцінки. Саме цей клас, не TypedDict, дає ValidationError."""

    setup: str = Field(description="Підводка")
    punchline: str = Field(description="Панчлайн")
    rating: Optional[int] = Field(default=None, ge=1, le=10, description="Ціле 1..10")


def robust_invoke(question: str, max_retries: int = 2) -> JokeModel:
    """Викликає модель і до двох разів повторює запит після помилки схеми.

    У текст повтору кладемо саму помилку, щоб модель бачила, яке поле зламалось.
    Початкове питання не затирається: кожна спроба дописує пояснення до нього, а не до
    вже роздутого попереднього промпта.

    Raises:
        ValidationError: об'єкт не пройшов Pydantic навіть на останній спробі.
        OutputParserException: відповідь не розібралась як об'єкт схеми.
    """
    structured = llm.with_structured_output(JokeModel)
    prompt = question
    last_error: Optional[Exception] = None

    for _ in range(max_retries + 1):
        try:
            return structured.invoke(prompt)
        except (ValidationError, OutputParserException) as error:
            last_error = error
            prompt = (
                f"{question}\n\n"
                f"Попередня відповідь не пройшла перевірку: {error}\n"
                "Поверни лише поля setup (str), punchline (str) і rating (int 1..10)."
            )
    if last_error is None:
        raise RuntimeError("robust_invoke завершився без результату і без помилки")
    raise last_error


Тестовий виклик. Збережений під клітинкою вивід — словник зі старого прогону, коли функція випадково сиділа на TypedDict і нічого не перевіряла. Повторний запуск надрукує об'єкт `JokeModel`.


In [ ]:
# Успішний виклик повертає JokeModel. Збережений нижче вивід — словник зі старого прогону.
print(robust_invoke("Tell me a joke about cats"))


{'setup': 'Why did the cat sit on the computer?', 'punchline': 'Because it wanted to keep an eye on the mouse!', 'rating': 8}


---

## Підсумок

У цьому ноутбуці ми розглянули:

- **LCEL** — спосіб з'єднувати компоненти через оператор `|`
- **Prompt Templates** — динамічні промпти зі змінними (`langchain_core.prompts`)
- **Chains** — ланцюг промпт → модель → парсер
- **Structured Output** — об'єкт фіксованої форми:
  - Pydantic (з валідацією; для Union, від якого залежить код, беріть саме його)
  - TypedDict (словник і стримінг, без перевірки)
  - JSON Schema (та сама ідея без класу Python)
  - Union-схеми (кілька варіантів відповіді)
  - Few-shot (приклади в системному промпті, дужки JSON подвоєні)
  - Повтор після `ValidationError` / `OutputParserException` лише навколо Pydantic

Паралельні гілки, `RunnableBranch`, LangSmith і наскрізний кейс — у `MU of AI_topic_5_lesson.ipynb` і в конспекті `5_langchain_ecosystem_notes.md`.
